# Baseline modele: LSTM i Transformer

Notebook definiuje pipeline do porównania baseline modeli dla rozpoznawania krótkich komend głosowych. Nie uruchamia eksperymentów automatycznie; konfiguracje, modele, dataset, trening i zapis wyników znajdują się w modułach `scripts`.

## Konfiguracja

Importy z lokalnego pakietu oraz podstawowe etykiety zadania.

In [22]:
from scripts import (
    DataFixedParams,
    DataGridParams,
    Experiment,
    FeatureFixedParams,
    FitFixedParams,
    FitGridParams,
    LABEL_ORDER,
    ModelGridParams,
    experiment_grid_dataframe,
    run_experiment,
)

LABEL_ORDER

('yes',
 'no',
 'up',
 'down',
 'left',
 'right',
 'on',
 'off',
 'stop',
 'go',
 'unknown',
 'silence')

## Interfejs eksperymentów

Eksperyment jest opisany przez dataclassy. Parametry stałe opisują dane, cechy i trening, a klasy `GridParams` pozwalają podać pojedynczą wartość albo listę wartości do porównania.

In [23]:
default_experiment = Experiment(name="default_full_data_baseline")
# experiment_grid_dataframe(default_experiment)

## Pipeline danych

Moduł `scripts.data` buduje manifest z `train.7z`, mapuje klasy spoza listy docelowej do `unknown`, tworzy przykłady `silence` z `_background_noise_` i przygotowuje mel-spektrogramy przez `torchaudio`. Oficjalny `testing_list.txt` jest używany jako test z etykietami; Kaggle `test.7z` nie służy do metryk, bo nie zawiera etykiet.

In [24]:
data_fixed = DataFixedParams()
feature_fixed = FeatureFixedParams()

data_fixed, feature_fixed

(DataFixedParams(data_dir='data', train_archive='train.7z', cache_dir='.cache/baseline_audio', output_dir='reports/02_baseline_models', target_labels=('yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go'), unknown_label='unknown', silence_label='silence', sample_rate=16000, clip_seconds=1.0, include_silence=True),
 FeatureFixedParams(n_mels=64, n_fft=512, hop_length=160, normalize=True))

## Modele baseline

Moduł `scripts.models` zawiera `LSTMBaseline` i `TransformerBaseline`. Oba modele przyjmują tę samą reprezentację wejściową: sekwencję ramek mel-spektrogramu w formacie `[czas, pasma_mel]`.

In [25]:
model_grid = ModelGridParams(
    model_type=["lstm", "transformer"],
    dropout=0.2,
    lstm_hidden_size=128,
    lstm_layers=2,
    lstm_bidirectional=True,
    transformer_d_model=128,
    transformer_heads=4,
    transformer_layers=2,
    transformer_ff_dim=256,
)

model_grid

ModelGridParams(model_type=['lstm', 'transformer'], dropout=0.2, lstm_hidden_size=128, lstm_layers=2, lstm_bidirectional=True, transformer_d_model=128, transformer_heads=4, transformer_layers=2, transformer_ff_dim=256)

## Wyniki i wykresy

Po uruchomieniu `run_experiment` każdy run zapisuje konfigurację, manifest, metryki, krzywe uczenia i macierz pomyłek do katalogu `reports/02_baseline_models/<experiment>/<run>/`. Postęp kontrolują pola `use_tqdm`, `progress_backend` i `verbose` w `FitFixedParams`.

In [26]:
fit_fixed = FitFixedParams(
    device="cpu",
    use_tqdm=True,
    progress_backend="terminal",
    verbose=True,
)
fit_grid = FitGridParams(
    epochs=3,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
)

fit_fixed, fit_grid

(FitFixedParams(device='cpu', num_workers=0, pin_memory=False, use_tqdm=True, progress_backend='terminal', verbose=True, log_every=20),
 FitGridParams(epochs=3, batch_size=64, learning_rate=0.001, weight_decay=0.0001))

## Definicja baseline eksperymentu

Domyślne klasy konfiguracyjne używają pełnych danych. Poniższy eksperyment startowy jest ustawiony na 5% splitu treningowego i 5% splitu testowego, żeby można było szybko sprawdzić pipeline po odkomentowaniu uruchomienia.

In [27]:
baseline_experiment = Experiment(
    name="baseline_lstm_transformer_5_percent",
    data_fixed=data_fixed,
    feature_fixed=feature_fixed,
    fit_fixed=fit_fixed,
    data_grid=DataGridParams(
        train_fraction=0.10,
        test_fraction=0.5,
        unknown_fraction=0.01,
        silence_examples_per_split=1200,
        sampling_strategy="natural",
        seed=42,
    ),
    model_grid=model_grid,
    fit_grid=fit_grid,
)

# experiment_grid_dataframe(baseline_experiment)

## Uruchomienie eksperymentu

Poniższa komórka jest celowo zakomentowana. Jej odkomentowanie uruchomi oba baseline modele i zapisze wyniki w strukturze katalogów opisanej wyżej.

In [ ]:
results = run_experiment(baseline_experiment)
results

Starting experiment: baseline_lstm_transformer_5_percent

Building dataset
  -> TRAIN split | samples: 2305
     class distribution:
       - down: 185
       - go: 187
       - left: 184
       - no: 186
       - off: 184
       - on: 187
       - right: 186
       - silence: 120
       - stop: 189
       - unknown: 326
       - up: 185
       - yes: 186
  -> TEST split | samples: 1929

Configuration run 1/2:
DATA (variable):
  - train_fraction: 0.1
  - test_fraction: 0.5
  - unknown_fraction: 0.01
  - silence_examples_per_split: 1200
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: lstm
  - dropout: 0.2
  - lstm_hidden_size: 128
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 128
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 256
FIT (variable):
  - epochs: 3
  - batch_size: 64
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cpu


Training:   0%|          | 0/3 [00:00<?, ?it/s]